In [ ]:
import json
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import pandas as pd

INPUT_FILE = Path("/Users/olgagavrilenko/DataspellProjects/Course-work/prepared_from_vk/user_graph_context.csv")
OUTPUT_FILE = Path("/Users/olgagavrilenko/DataspellProjects/Course-work/prepared_from_vk/edges_ready.csv")

# Ограничения на "слишком широкие" общие сущности,
# чтобы граф не стал слишком плотным
MAX_GROUP_USERS = 300
MAX_CITY_USERS = 300
MAX_UNIVERSITY_USERS = 300
MAX_FACULTY_USERS = 300


def parse_json_list(x):
    if pd.isna(x):
        return set()

    if isinstance(x, list):
        return set(int(v) for v in x)

    if isinstance(x, str):
        x = x.strip()
        if not x:
            return set()
        try:
            data = json.loads(x)
            if isinstance(data, list):
                return set(int(v) for v in data)
        except Exception:
            return set()

    return set()


def load_user_graph_context(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path.resolve()}")

    df = pd.read_csv(path)

    required_cols = [
        "vk_id",
        "friends_json",
        "groups_json",
        "city_id",
        "university",
        "faculty",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"В файле нет нужных колонок: {missing}")

    df["vk_id"] = pd.to_numeric(df["vk_id"], errors="coerce")
    df = df.dropna(subset=["vk_id"]).copy()
    df["vk_id"] = df["vk_id"].astype(int)

    df = df.drop_duplicates(subset=["vk_id"], keep="last").copy()
    return df


def build_maps(df: pd.DataFrame):
    friends_map = {}
    groups_map = {}
    city_map = {}
    education_map = {}

    for _, row in df.iterrows():
        uid = int(row["vk_id"])

        friends_map[uid] = parse_json_list(row["friends_json"])
        groups_map[uid] = parse_json_list(row["groups_json"])

        city_val = row["city_id"]
        if pd.isna(city_val):
            city_val = None
        else:
            try:
                city_val = int(city_val)
            except Exception:
                city_val = None
        city_map[uid] = city_val

        university = row["university"]
        faculty = row["faculty"]

        if pd.isna(university):
            university = None
        if pd.isna(faculty):
            faculty = None

        education_map[uid] = {
            "university": university,
            "faculty": faculty,
        }

    return friends_map, groups_map, city_map, education_map


def add_pair(candidate_pairs: set, u1: int, u2: int):
    if u1 == u2:
        return
    a, b = sorted((u1, u2))
    candidate_pairs.add((a, b))


def build_candidate_pairs(
        vk_ids,
        friends_map,
        groups_map,
        city_map,
        education_map,
        max_group_users=300,
        max_city_users=300,
        max_university_users=300,
        max_faculty_users=300,
):
    candidate_pairs = set()

    # 1. Прямые друзья
    for u1 in vk_ids:
        for u2 in friends_map.get(u1, set()):
            if u2 in friends_map:
                add_pair(candidate_pairs, u1, u2)

    print(f"После дружбы: {len(candidate_pairs)} кандидатных пар")

    # 2. Общие группы
    users_by_group = defaultdict(list)
    for uid, group_ids in groups_map.items():
        for gid in group_ids:
            users_by_group[gid].append(uid)

    for gid, users in users_by_group.items():
        if 2 <= len(users) <= max_group_users:
            for u1, u2 in combinations(users, 2):
                add_pair(candidate_pairs, u1, u2)

    print(f"После групп: {len(candidate_pairs)} кандидатных пар")

    # 3. Общий город
    users_by_city = defaultdict(list)
    for uid, city_id in city_map.items():
        if city_id is not None:
            users_by_city[city_id].append(uid)

    for city_id, users in users_by_city.items():
        if 2 <= len(users) <= max_city_users:
            for u1, u2 in combinations(users, 2):
                add_pair(candidate_pairs, u1, u2)

    print(f"После городов: {len(candidate_pairs)} кандидатных пар")

    # 4. Общий университет
    users_by_university = defaultdict(list)
    for uid, edu in education_map.items():
        univ = edu.get("university")
        if univ is not None:
            users_by_university[univ].append(uid)

    for univ, users in users_by_university.items():
        if 2 <= len(users) <= max_university_users:
            for u1, u2 in combinations(users, 2):
                add_pair(candidate_pairs, u1, u2)

    print(f"После университетов: {len(candidate_pairs)} кандидатных пар")

    # 5. Общий факультет
    users_by_faculty = defaultdict(list)
    for uid, edu in education_map.items():
        fac = edu.get("faculty")
        if fac is not None:
            users_by_faculty[fac].append(uid)

    for fac, users in users_by_faculty.items():
        if 2 <= len(users) <= max_faculty_users:
            for u1, u2 in combinations(users, 2):
                add_pair(candidate_pairs, u1, u2)

    print(f"После факультетов: {len(candidate_pairs)} кандидатных пар")

    return candidate_pairs

def build_edges_expanded_from_candidates(
        candidate_pairs,
        friends_map,
        groups_map,
        city_map,
        education_map,
):
    rows = []

    for idx, (u1, u2) in enumerate(sorted(candidate_pairs), start=1):
        if idx % 100000 == 0:
            print(f"Обработано кандидатных пар: {idx}")

        mutual_friends = len(friends_map.get(u1, set()) & friends_map.get(u2, set()))
        mutual_groups = len(groups_map.get(u1, set()) & groups_map.get(u2, set()))

        common_city = int(
            city_map.get(u1) is not None and
            city_map.get(u1) == city_map.get(u2)
        )

        common_university = int(
            education_map.get(u1, {}).get("university") is not None and
            education_map.get(u1, {}).get("university") == education_map.get(u2, {}).get("university")
        )

        common_faculty = int(
            education_map.get(u1, {}).get("faculty") is not None and
            education_map.get(u1, {}).get("faculty") == education_map.get(u2, {}).get("faculty")
        )

        # приближённый friend_status
        friend_status = int(
            (u2 in friends_map.get(u1, set())) or
            (u1 in friends_map.get(u2, set()))
        )

        # Та же логика, что в репозитории:
        # ребро существует, если хотя бы один edge feature > 0
        if any([
            mutual_friends > 0,
            mutual_groups > 0,
            common_city > 0,
            common_university > 0,
            common_faculty > 0,
            friend_status > 0,
        ]):
            rows.append({
                "source": u1,
                "target": u2,
                "mutual_friends": mutual_friends,
                "mutual_groups": mutual_groups,
                "common_city": common_city,
                "common_university": common_university,
                "common_faculty": common_faculty,
                "friend_status": friend_status,
            })

    return pd.DataFrame(rows)


def main():
    print("1. Загружаю контекст пользователей...")
    df = load_user_graph_context(INPUT_FILE)

    vk_ids = df["vk_id"].tolist()
    print(f"Пользователей во входном файле: {len(vk_ids)}")

    print("2. Собираю словари для графа...")
    friends_map, groups_map, city_map, education_map = build_maps(df)

    print("3. Формирую кандидатные пары...")
    candidate_pairs = build_candidate_pairs(
        vk_ids=vk_ids,
        friends_map=friends_map,
        groups_map=groups_map,
        city_map=city_map,
        education_map=education_map,
        max_group_users=MAX_GROUP_USERS,
        max_city_users=MAX_CITY_USERS,
        max_university_users=MAX_UNIVERSITY_USERS,
        max_faculty_users=MAX_FACULTY_USERS,
    )

    print(f"Итого кандидатных пар: {len(candidate_pairs)}")

    print("4. Считаю признаки рёбер...")
    edges_expanded = build_edges_expanded_from_candidates(
        candidate_pairs=candidate_pairs,
        friends_map=friends_map,
        groups_map=groups_map,
        city_map=city_map,
        education_map=education_map,
    )

    print(f"Рёбер после фильтрации: {len(edges_expanded)}")

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    edges_expanded.to_csv(OUTPUT_FILE, index=False)

    print("Готово.")
    print(OUTPUT_FILE)


if __name__ == "__main__":
    main()